# 🛡️ APKSHIELD — live dashboard on Google Colab

Run the full backend **and** dashboard with zero local setup: **Runtime → Run all**, then click the URL printed by the last cell.

- Paste your free API keys in the **Keys** cell for the full experience (Claude report + reverse engineering, VirusTotal + MalwareBazaar threat-intel + dynamic sandbox). Leave them blank to run in offline/template mode — the static + ML + impersonation + risk pipeline still works fully.
- Optional: fetch real in-the-wild banking-trojan samples (needs a MalwareBazaar key).

In [ ]:
#@title 1 · Clone repo + install dependencies (~2 min)
import os, subprocess, sys
REPO = "https://github.com/Dexterous-Ruler/apk.git"  #@param {type:"string"}
if not os.path.isdir("apk"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "apk"], check=True)
os.chdir("/content/apk")
print("installing python deps...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("done. cwd =", os.getcwd())

In [ ]:
#@title 2 · API keys (optional — blank = offline/template mode)
ANTHROPIC_API_KEY = ""  #@param {type:"string"}
VT_API_KEY = ""  #@param {type:"string"}
MWB_API_KEY = ""  #@param {type:"string"}
with open(".env", "w") as f:
    for k, v in [("ANTHROPIC_API_KEY", ANTHROPIC_API_KEY),
                 ("VT_API_KEY", VT_API_KEY), ("MWB_API_KEY", MWB_API_KEY)]:
        if v.strip():
            f.write(f"{k}={v.strip()}\n")
print("keys set:", [k for k, v in [('ANTHROPIC', ANTHROPIC_API_KEY), ('VT', VT_API_KEY), ('MWB', MWB_API_KEY)] if v.strip()] or "(none — offline mode)")

In [ ]:
#@title 3 · (optional) fetch real banking-trojan samples — needs MalwareBazaar key
import subprocess, sys, os
if os.path.exists(".env") and "MWB_API_KEY" in open(".env").read():
    print("downloading a few real samples (encrypted, analyzed in-memory only)...")
    subprocess.run([sys.executable, "analysis/fetch_real_samples.py", "--tag", "Cerberus", "--limit", "3", "--llm", "template"])
else:
    print("no MWB_API_KEY — skipping. The crafted + benign samples still demo the full pipeline.")

In [ ]:
#@title 4 · Launch the dashboard 🚀  (click the URL it prints)
import subprocess, sys, time, urllib.request, os
os.environ["APK_HOST"] = "127.0.0.1"; os.environ["APK_PORT"] = "8800"
proc = subprocess.Popen([sys.executable, "server.py"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
# wait until the server answers
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8800/api/config", timeout=2); break
    except Exception:
        time.sleep(1)
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8800)")
print("\n\u2705 APKSHIELD is live — open this URL:\n\n   " + url + "\n")
print("Tip: click a sample chip, or a REAL · Cerberus chip then 'Run deep analysis' for the reverse-engineering + dynamic showcase.")